# Fraud Detection Analytics

**CODTECH IT Solutions — Data Analytics Internship — Task 4**

Intern: Chakiri Subramanyam | Intern ID: CITS5137

## 1. Generate the Dataset

In [ ]:
"""
generate_data.py
Generates a synthetic financial transactions dataset for the Fraud Detection
Analytics task (Codtech IT Solutions - Data Analytics Internship, Task 4).
"""

import numpy as np
import pandas as pd

np.random.seed(42)

N = 5000

transaction_id = [f"TXN{i:06d}" for i in range(1, N + 1)]
date_range = pd.to_datetime(
    np.random.randint(
        pd.Timestamp("2024-01-01").value,
        pd.Timestamp("2024-12-31").value,
        N
    )
)

transaction_type = np.random.choice(
    ["Online Purchase", "ATM Withdrawal", "Bank Transfer", "POS Payment", "International"],
    N, p=[0.35, 0.20, 0.20, 0.15, 0.10]
)

amount = np.where(
    np.random.rand(N) < 0.05,
    np.random.uniform(3000, 15000, N),   # large suspicious amounts
    np.random.exponential(scale=250, size=N) + 10
).round(2)

hour = np.random.randint(0, 24, N)
location = np.random.choice(
    ["Domestic", "International"], N, p=[0.80, 0.20]
)
card_type = np.random.choice(["Credit", "Debit"], N, p=[0.55, 0.45])
num_transactions_day = np.random.poisson(3, N)
account_age_years = np.random.uniform(0.1, 15, N).round(2)
failed_attempts = np.random.choice([0, 1, 2, 3], N, p=[0.80, 0.10, 0.06, 0.04])
distance_from_home_km = np.random.exponential(scale=50, size=N).round(2)

# Build fraud probability
fraud_prob = (
    0.01
    + 0.20 * (amount > 3000)
    + 0.15 * (location == "International")
    + 0.10 * ((hour >= 0) & (hour <= 4))
    + 0.12 * (transaction_type == "International")
    + 0.10 * (num_transactions_day > 7)
    + 0.08 * (failed_attempts > 0)
    + 0.05 * (distance_from_home_km > 200)
    - 0.01 * account_age_years
)
fraud_prob = np.clip(fraud_prob, 0.01, 0.95)
is_fraud = np.random.binomial(1, fraud_prob)

df = pd.DataFrame({
    "TransactionID": transaction_id,
    "Date": date_range.strftime("%Y-%m-%d"),
    "Hour": hour,
    "TransactionType": transaction_type,
    "Amount": amount,
    "CardType": card_type,
    "Location": location,
    "Num_Transactions_Day": num_transactions_day,
    "Account_Age_Years": account_age_years,
    "Failed_Attempts": failed_attempts,
    "Distance_From_Home_km": distance_from_home_km,
    "Is_Fraud": is_fraud,
})

df.to_csv("data/transactions.csv", index=False)
print(f"Generated {len(df):,} rows -> data/transactions.csv")
print(f"Fraud rate: {is_fraud.mean()*100:.2f}%")


## 2. EDA, Model Training & Fraud Detection Analysis

In [ ]:
"""
fraud_detection_analytics.py
Task 4: Fraud Detection Analytics
Codtech IT Solutions - Data Analytics Internship

Loads the transactions dataset, performs EDA, trains fraud detection models
(Logistic Regression + Random Forest), evaluates performance, and saves
all visualizations to output/.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "data/transactions.csv"
OUT_DIR = "output"

# ---------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df.head())
print("\nFraud distribution:\n", df["Is_Fraud"].value_counts())

# ---------------------------------------------------------------
# 2. Fraud distribution
# ---------------------------------------------------------------
fraud_counts = df["Is_Fraud"].map({0: "Legitimate", 1: "Fraud"}).value_counts()

plt.figure(figsize=(6, 5))
sns.barplot(x=fraud_counts.index, y=fraud_counts.values,
            hue=fraud_counts.index, palette=["#2ecc71", "#e74c3c"], legend=False)
plt.title("Transaction Distribution: Fraud vs Legitimate", fontsize=14, fontweight="bold")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_fraud_distribution.png")
plt.close()

# ---------------------------------------------------------------
# 3. Fraud by transaction type
# ---------------------------------------------------------------
fraud_type = df.groupby("TransactionType")["Is_Fraud"].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(9, 5))
sns.barplot(x=fraud_type.index, y=fraud_type.values, hue=fraud_type.index,
            palette="Reds_d", legend=False)
plt.title("Fraud Rate (%) by Transaction Type", fontsize=14, fontweight="bold")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_fraud_by_transaction_type.png")
plt.close()

# ---------------------------------------------------------------
# 4. Amount distribution: Fraud vs Legitimate
# ---------------------------------------------------------------
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="Amount", hue=df["Is_Fraud"].map({0: "Legitimate", 1: "Fraud"}),
             bins=50, kde=True, palette=["#2ecc71", "#e74c3c"])
plt.xlim(0, 3000)
plt.title("Transaction Amount Distribution: Fraud vs Legitimate", fontsize=14, fontweight="bold")
plt.xlabel("Amount ($)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_amount_distribution.png")
plt.close()

# ---------------------------------------------------------------
# 5. Fraud by hour of day
# ---------------------------------------------------------------
fraud_hour = df.groupby("Hour")["Is_Fraud"].mean() * 100

plt.figure(figsize=(11, 5))
plt.bar(fraud_hour.index, fraud_hour.values, color="#e74c3c", alpha=0.8)
plt.title("Fraud Rate (%) by Hour of Day", fontsize=14, fontweight="bold")
plt.xlabel("Hour of Day")
plt.ylabel("Fraud Rate (%)")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_fraud_by_hour.png")
plt.close()

# ---------------------------------------------------------------
# 6. Preprocessing
# ---------------------------------------------------------------
model_df = df.drop(columns=["TransactionID", "Date"]).copy()

for col in ["TransactionType", "CardType", "Location"]:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col])

X = model_df.drop(columns=["Is_Fraud"])
y = model_df["Is_Fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# ---------------------------------------------------------------
# 7. Train models
# ---------------------------------------------------------------
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                             max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print("\n--- Logistic Regression ---")
print("Accuracy:", round(accuracy_score(y_test, lr_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, lr_proba), 4))
print(classification_report(y_test, lr_pred))

print("\n--- Random Forest ---")
print("Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_proba), 4))
print(classification_report(y_test, rf_pred))

# ---------------------------------------------------------------
# 8. Confusion matrix
# ---------------------------------------------------------------
cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Legitimate", "Fraud"], yticklabels=["Legitimate", "Fraud"])
plt.title("Confusion Matrix — Random Forest", fontsize=14, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/05_confusion_matrix.png")
plt.close()

# ---------------------------------------------------------------
# 9. ROC curve
# ---------------------------------------------------------------
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC={roc_auc_score(y_test, lr_proba):.2f})", linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={roc_auc_score(y_test, rf_proba):.2f})", linewidth=2)
plt.plot([0, 1], [0, 1], "k--")
plt.title("ROC Curve — Fraud Detection", fontsize=14, fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/06_roc_curve.png")
plt.close()

# ---------------------------------------------------------------
# 10. Precision-Recall curve
# ---------------------------------------------------------------
prec_rf, rec_rf, _ = precision_recall_curve(y_test, rf_proba)
ap_rf = average_precision_score(y_test, rf_proba)

plt.figure(figsize=(7, 6))
plt.plot(rec_rf, prec_rf, color="#e74c3c", linewidth=2,
         label=f"Random Forest (AP={ap_rf:.2f})")
plt.title("Precision-Recall Curve — Fraud Detection", fontsize=14, fontweight="bold")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/07_precision_recall_curve.png")
plt.close()

# ---------------------------------------------------------------
# 11. Feature importance
# ---------------------------------------------------------------
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index,
            palette="flare", legend=False)
plt.title("Feature Importance — Random Forest Fraud Detector", fontsize=14, fontweight="bold")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/08_feature_importance.png")
plt.close()

# ---------------------------------------------------------------
# 12. Save summary
# ---------------------------------------------------------------
summary = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, lr_pred), accuracy_score(y_test, rf_pred)],
    "ROC_AUC": [roc_auc_score(y_test, lr_proba), roc_auc_score(y_test, rf_proba)],
    "Avg_Precision": [average_precision_score(y_test, lr_proba), ap_rf],
})
summary.to_csv(f"{OUT_DIR}/model_summary.csv", index=False)

print("\nAll charts saved to output/")
print(summary)
